### Pick WiFI sessions that have better decoding over null
- Stimulus
- Choice (?)

In [5]:
from glob import glob
import pickle as pkl
import numpy as np
from matplotlib import pyplot as plt
import re
import numpy as np
from statsmodels.stats.multitest import multipletests

In [6]:
# find real and null results
# and then choose only sessions/animals/regions that are better than null

In [7]:
from collections import defaultdict
import numpy as np


def get_significant_masks(true_dict, null_dict, percentile=95):
    """
    Args:
        true_dict: Dict {Frame_Key -> {Pair_Key -> [List of Bootstraps]}}
                   Where each bootstrap is {'prob_acc_a': float, 'prob_acc_b': float}
        null_dict: Same structure (shuffled labels).
        percentile: 95 (default) checks if True Mean > 95th Percentile of Null.

    Returns:
        mask_a, mask_b: Dicts with same keys as input, containing True/False.
    """

    sig_regions_by_frame = {}
    frame_results = {}
    for frame_key in true_dict.keys():
        frame_significant_set = set()
        region_gains = defaultdict(list)

        for pair_key in true_dict[frame_key].keys():

            parts = pair_key.split("_")
            reg_name_a = parts[0].strip("[]'")
            reg_name_b = parts[1].strip("[]'")

            true_bootstraps = true_dict[frame_key][pair_key]["results"]
            null_bootstraps = null_dict[frame_key][pair_key]

            vals_true_a = [b["balanced_acc_A"] for b in true_bootstraps]
            vals_null_a = null_bootstraps["acc_A"]
            thresh_a = np.percentile(vals_null_a, percentile)
            is_sig_a = np.mean(vals_true_a) > thresh_a
            if is_sig_a:
                gain = np.mean(vals_true_a) - np.mean(vals_null_a)
                frame_significant_set.add(reg_name_a)
                region_gains[reg_name_a].append(gain)

            vals_true_b = [b["balanced_acc_B"] for b in true_bootstraps]
            vals_null_b = null_bootstraps["acc_B"]

            thresh_b = np.percentile(vals_null_b, percentile)
            is_sig_b = np.mean(vals_true_b) > thresh_b

            if is_sig_b:
                gain = np.mean(vals_true_b) - np.mean(vals_null_b)
                frame_significant_set.add(reg_name_b)
                region_gains[reg_name_b].append(gain)

        avg_gains_for_frame = {reg: np.mean(gains) for reg, gains in region_gains.items()}
        sig_regions_by_frame[frame_key] = frame_significant_set
        frame_results[frame_key] = avg_gains_for_frame
    return sig_regions_by_frame, frame_results

In [8]:
files_stim = np.sort(glob("../data/generated/wfi_decoders/stim/equi_3/*.pkl"))
files_choice = np.sort(glob("../data/generated/wfi_decoders/choice/equi_3/*.pkl"))
files_stim_null = np.sort(glob("../data/generated/wfi_decoders/stim/null/*.pkl"))
files_choice_null = np.sort(glob("../data/generated/wfi_decoders/choice/null/*.pkl"))

In [9]:
# sanity check
for idx in range(len(files_stim)):
    eid_stim = files_stim[idx].rsplit("/")[-1].rsplit("_wfi")[0]
    eid_stim_null = files_stim_null[idx].rsplit("/")[-1].rsplit("_wfi")[0]
    assert eid_stim == eid_stim_null

In [10]:
# now what
regions_per_eid = {}
region_gains = defaultdict(list)
for idx in range(len(files_stim)):
    eid_stim = files_stim[idx].rsplit("/")[-1].rsplit("_wfi")[0]
    with open(files_stim[idx], "rb") as f:
        d1 = pkl.load(f)
    with open(files_stim_null[idx], "rb") as f:
        d2 = pkl.load(f)

    sig_region_by_frame, frame_gains = get_significant_masks(d1, d2)

    # i only care about frame 2
    for k in frame_gains[1].keys():
        region_gains[k].append(frame_gains[1][k])
    regions_per_eid[eid_stim] = list(sig_region_by_frame[1])

In [105]:
avg_gains_for_frame = {reg: [np.mean(gains), np.std(gains)] for reg, gains in region_gains.items()}

In [29]:
# with open("../data/processed/wifi_stim_good_regions_per_sessions_frame1.pkl", "wb") as f:
#     pkl.dump(regions_per_eid, f)

with open(
    "../data/processed/wfi_stim_frames/wifi_stim_good_regions_per_sessions_frame2.pkl", "rb"
) as f:
    data = pkl.load(f)

In [30]:
list_of_regions = list(regions_per_eid.values())

In [113]:
from collections import Counter


total_counts = Counter()

for animal, frame_dict in regions_per_eid.items():
    # Get all unique regions that were significant in ANY frame for this animal
    total_counts.update(frame_dict)